# 06 Results

Reads the saved answers in `results/*_final.csv` and computes every number reported in the paper.

In [1]:
import json
from pathlib import Path

import pandas as pd

from ragtrust.analysis import (add_knowledge, answer_kinds, baseline_table, condition_tables, dose_table, flippable,
                               knowledge_setup, load_controlled, load_retrieval, primary_tests, retrieval_tables, tag,
                               weighted_unknown)
from ragtrust.poison import load_poison_set
from ragtrust.runs import load_selection

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
RES, TABLES, FIGS = ROOT / "results", ROOT / "results" / "tables", ROOT / "results" / "figures"
TABLES.mkdir(exist_ok=True), FIGS.mkdir(exist_ok=True)
MODELS = ["llama3.1:8b", "qwen2.5:7b"]
POPULATION = {"A": 556, "B": 3372}   # poisonable questions known by >= 1 model (A) / by neither (B), notebook 00
pd.set_option("display.width", 200)


def save(df, name):
    df.to_csv(TABLES / f"{name}.csv")
    return df

## 1. Data and knowledge groups

The 751 generated poison items minus the 81 judged bad in the hand check = 670 items.
For each model, every question falls into one of four cells: known by **both** models, by **this** model only,
by the **other** model only, or by **neither**. **Weights:** stratum A (known by at least one model) and stratum B
(known by neither) were sampled at different rates, so each question stands for population size / sample size questions
of its stratum.

In [2]:
examples = load_selection()
by_id = {ex.id: ex for ex in examples}
items = load_poison_set(ROOT / "data" / "poison" / "gemma4-12b_final.jsonl")
item_by_id = {it.id: it for it in items}
excluded = set(json.load(open(ROOT / "data" / "poison" / "excluded.json", encoding="utf-8")))
item_ids = [it.id for it in items if it.id not in excluded]

selection = {m: pd.read_csv(RES / f"selection_closed_book_{tag(m)}_all.csv") for m in MODELS}
know = knowledge_setup(selection, item_ids, MODELS, POPULATION)
print("poison items:", len(item_ids), "| weights:", know.groupby("stratum")["weight"].first().round(2).to_dict())
save(pd.DataFrame({m: know[f"cell_{m}"].value_counts() for m in MODELS}), "knowledge_cells")

poison items: 670 | weights: {'A': 1.67, 'B': 10.01}


,llama3.1:8b,qwen2.5:7b
both,114,114
neither,337,337
other_only,70,149
this_only,149,70


## 2. Baselines (no poison)

Closed-book, the 10 dataset paragraphs, and the 2 gold paragraphs. `match_A` / `match_B` per stratum; the weighted
columns are the estimates for all poisonable HotpotQA questions (EM and F1 as in the HotpotQA literature).

In [3]:
stratum_of = pd.Series(json.load(open(ROOT / "data" / "poison" / "final_selection.json"))["group"]).map({"known": "A", "unknown": "B"})
save(baseline_table(RES, MODELS, stratum_of, POPULATION), "baselines")

match_A  match_B  match_weighted  em_weighted  f1_weighted
model       setting                                                                
llama3.1:8b closed_book    0.750    0.000           0.106        0.098        0.160
            distractor     0.845    0.500           0.549        0.485        0.630
            gold_only      0.925    0.610           0.655        0.584        0.746
qwen2.5:7b  closed_book    0.555    0.000           0.079        0.070        0.140
            distractor     0.790    0.475           0.520        0.459        0.610
            gold_only      0.908    0.592           0.637        0.554        0.725

## 3. Scoring

All controlled-poisoning answers (4 conditions × 2 models × m = 0..4), re-scored with the lenient rule.
Rows: the category stored at run time (strict rule); columns: the lenient category.

In [4]:
P = add_knowledge(load_controlled(RES, MODELS, item_ids), know, MODELS)
save(pd.crosstab(P["category_strict"], P["category"]), "rescoring_strict_vs_lenient")

category,correct,hedged,other,poisoned
category_strict,,,,
correct,13592,0,0,0
hedged,0,144,0,0
other,1977,0,2567,600
poisoned,0,0,0,7920


## 4. Dose-response

**Denominator:** per model, the questions answered correctly at m = 0 in all four conditions (only these can be flipped).

In [5]:
K = flippable(P)
base = K[K["condition"] == "baseline"]
print("questions kept:", base[base["m"] == 0].groupby("model").size().to_dict())

dose_cells = save(dose_table(base, ["model", "cell"]).round(3).set_index(["model", "cell", "m"]), "dose_by_cell")
dose_groups = dose_table(base, ["model", "group"])          # for figure 1
save(weighted_unknown(base, MODELS), "dose_unknown_weighted")
dose_cells["poisoned"].unstack("m")

questions kept: {'llama3.1:8b': 599, 'qwen2.5:7b': 601}


m                         0      1      2      3      4
model       cell                                       
llama3.1:8b both        0.0  0.018  0.119  0.431  0.752
            neither     0.0  0.101  0.290  0.559  0.843
            other_only  0.0  0.095  0.206  0.540  0.778
            this_only   0.0  0.050  0.156  0.433  0.773
qwen2.5:7b  both        0.0  0.028  0.241  0.583  0.889
            neither     0.0  0.048  0.328  0.631  0.898
            other_only  0.0  0.029  0.250  0.618  0.860
            this_only   0.0  0.078  0.281  0.656  0.906

## 5. The five primary hypotheses

Tested at m = 2 except H1.

`share_1` / `share_2`: poisoned share in the first / second condition; `only_1` / `only_2`: discordant questions.

In [6]:
tests = save(primary_tests(K, MODELS).round(4), "tests")
tests

,n,share_1,share_2,diff_2_minus_1,ci_lo,ci_hi,only_1,only_2,p,p_holm
comparison,,,,,,,,,,
H1 dose: m=1 vs m=3 | llama3.1:8b,599,0.0735,0.5042,0.4307,0.3940,0.4708,4,262,0.0000,0.0000
"H2 baseline vs consistency prompt, m=2 | llama3.1:8b",599,0.2187,0.2154,-0.0033,-0.0217,0.0150,17,15,0.8601,1.0000
"H3 poison labelled untrusted vs trusted, m=2 | llama3.1:8b",599,0.1452,0.3055,0.1603,0.1319,0.1903,2,98,0.0000,0.0000
"H4 known vs unknown-by-neither, m=2 | llama3.1:8b",536,0.1400,0.2902,0.1502,NaN,NaN,250,286,0.0000,0.0001
H1 dose: m=1 vs m=3 | qwen2.5:7b,601,0.0433,0.6223,0.5790,0.5391,0.6173,2,350,0.0000,0.0000
"H2 baseline vs consistency prompt, m=2 | qwen2.5:7b",601,0.2895,0.2829,-0.0067,-0.0250,0.0116,19,15,0.6076,1.0000
"H3 poison labelled untrusted vs trusted, m=2 | qwen2.5:7b",601,0.2196,0.4027,0.1830,0.1531,0.2180,5,115,0.0000,0.0000
"H4 known vs unknown-by-neither, m=2 | qwen2.5:7b",465,0.2558,0.3276,0.0718,NaN,NaN,172,293,0.1165,0.3495
"H5 llama3.1:8b vs qwen2.5:7b, m=2 | common questions",574,0.2178,0.2909,0.0732,0.0314,0.1150,57,99,0.0010,0.0039


## 6. Prompt and source labels

Poisoned share by m for the four conditions (same questions within a model). The labels can only matter at m = 1-3:
at m = 0 and m = 4 every version that states an answer carries the same label. Hedged answers (true and false answer both given) are shown below.

In [7]:
conditions, hedged = condition_tables(K)
save(hedged, "hedged_by_condition")
save(conditions, "conditions_sample")

m                          0      1      2      3      4
model       condition                                   
llama3.1:8b baseline     0.0  0.073  0.219  0.504  0.803
            consistency  0.0  0.078  0.215  0.459  0.770
            trusted      0.0  0.132  0.306  0.508  0.795
            untrusted    0.0  0.065  0.145  0.336  0.773
qwen2.5:7b  baseline     0.0  0.043  0.290  0.622  0.889
            consistency  0.0  0.047  0.283  0.624  0.895
            trusted      0.0  0.083  0.403  0.671  0.885
            untrusted    0.0  0.030  0.220  0.547  0.875

## 7. Retrieval

The top 5 paragraphs come from BM25 or dense retrieval over all paragraphs of the 800 questions, in which the original answer
paragraph is **replaced** by its 4 versions (so at m = 4 the true paragraph is not in the collection).
First table: what was retrieved. Second: poisoned share per retriever, among questions correct at m = 0 with that retriever.

In [8]:
R = load_retrieval(RES, MODELS, item_ids)
retrieved, end_to_end = retrieval_tables(R, MODELS)
save(retrieved, "retrieval_only")

n_false  n_true  bridge_retrieved  found
retriever m                                          
bm25      0    0.000   1.860             0.775  0.642
          1    0.375   1.434             0.779  0.640
          2    0.752   0.984             0.779  0.637
          3    1.154   0.507             0.784  0.628
          4    1.584   0.000             0.781  0.561
dense     0    0.000   2.700             0.858  0.827
          1    0.675   2.013             0.860  0.836
          2    1.282   1.367             0.858  0.837
          3    1.942   0.681             0.858  0.834
          4    2.585   0.000             0.855  0.794

In [9]:
save(end_to_end, "retrieval_end_to_end")

n  poisoned  correct  n_found  poisoned_if_found
model       retriever m                                                    
llama3.1:8b bm25      0  456     0.000    1.000      390              0.000
                      1  456     0.053    0.921      385              0.060
                      2  456     0.125    0.811      380              0.147
                      3  456     0.298    0.612      371              0.367
                      4  456     0.496    0.342      324              0.694
            dense     0  525     0.000    1.000      495              0.000
                      1  525     0.091    0.876      495              0.097
                      2  525     0.213    0.741      494              0.227
                      3  525     0.408    0.531      489              0.436
                      4  525     0.684    0.210      463              0.765
qwen2.5:7b  bm25      0  429     0.000    1.000      383              0.000
                      1  429     0.047    0.925      377              0.053
                      2  429     0.193    0.753      372              0.223
                      3  429     0.396    0.527      364              0.467
                      4  429     0.590    0.247      315              0.790
            dense     0  515     0.000    1.000      494              0.000
                      1  515     0.107    0.864      493              0.112
                      2  515     0.249    0.705      491              0.261
                      3  515     0.528    0.419      487              0.556
                      4  515     0.744    0.153      460              0.824

## 8. Figures (saved to `results/figures/`)

In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e6", "savefig.dpi": 200, "savefig.bbox": "tight", "legend.frameon": False})
# colour + marker + line style, so a series is never identified by colour alone
STYLE = {"known": ("#0072B2", "o", "-"), "unknown (neither knows)": ("#D55E00", "s", "--"),
         "unknown (other model knows)": ("#009E73", "^", ":"), "baseline": ("#444444", "o", "-"),
         "consistency": ("#0072B2", "D", "--"), "trusted": ("#D55E00", "s", "-."), "untrusted": ("#009E73", "^", ":"),
         "bm25": ("#0072B2", "o", "-"), "dense": ("#D55E00", "s", "--"), "controlled": ("#444444", "D", ":")}
LABEL = {"consistency": "consistency prompt", "trusted": "false versions labelled Wikipedia",
         "untrusted": "false versions labelled forum"}
YLABEL = "share of answers = false answer"


def line(ax, x, y, key, label):
    c, mk, ls = STYLE[key]
    ax.plot(x, y, color=c, marker=mk, ls=ls, lw=1.6, ms=5, label=label)


def two_panels(name, draw):
    fig, axes = plt.subplots(1, 2, figsize=(7, 2.9), sharey=True)
    for ax, m in zip(axes, MODELS):
        draw(ax, m)
        ax.set(title=m, xticks=[0, 1, 2, 3, 4], xlabel="false versions m (of 4)", ylim=(-0.02, 1.25),
               yticks=[0, 0.2, 0.4, 0.6, 0.8, 1.0])     # room for the legend above the curves
        ax.legend(fontsize=7, loc="upper left")
    axes[0].set_ylabel(YLABEL)
    for ext in ["png", "pdf"]:
        fig.savefig(FIGS / f"{name}.{ext}")
    plt.close(fig)


def fig1(ax, m):
    for grp in ["known", "unknown (neither knows)", "unknown (other model knows)"]:
        g = dose_groups[(dose_groups["model"] == m) & (dose_groups["group"] == grp)].sort_values("m")
        c, mk, ls = STYLE[grp]
        ax.errorbar(g["m"], g["poisoned"], yerr=[(g["poisoned"] - g["lo"]).clip(lower=0), (g["hi"] - g["poisoned"]).clip(lower=0)],
                    color=c, marker=mk, ls=ls, lw=1.6, ms=5, capsize=2, label=f"{grp} (n={int(g['n'].iloc[0])})")


def fig2(ax, m):
    for cond, row in conditions.loc[m].iterrows():
        line(ax, row.index, row.values, cond, LABEL.get(cond, cond))


def fig3(ax, m):
    for r in ["bm25", "dense"]:
        g = end_to_end.loc[(m, r)]
        line(ax, g.index, g["poisoned"], r, f"{r} top-5 (n={int(g['n'].iloc[0])})")
    g = base[base["model"] == m].groupby("m")["poisoned"].mean()
    line(ax, g.index, g.values, "controlled", "controlled (all 4 versions shown)")


two_panels("fig1_dose_response", fig1)
two_panels("fig2_prompt_and_labels", fig2)
two_panels("fig3_retrieval", fig3)

print(sorted(p.name for p in FIGS.iterdir()))

['archive', 'fig1_dose_response.pdf', 'fig1_dose_response.png', 'fig2_prompt_and_labels.pdf', 'fig2_prompt_and_labels.png', 'fig3_retrieval.pdf', 'fig3_retrieval.png']
